# Proyecto RAG con LLM y métricas de evaluación

Este notebook implementa un **sistema RAG (Retrieval-Augmented Generation)** que combina:

1. **Carga y procesamiento de documentos** (PDFs reales).
2. **Chunking y generación de embeddings** para cada fragmento.
3. **Búsqueda semántica con FAISS HNSW** y BM25 híbrido.
4. **Reranking con Cross-Encoder** para mejorar precisión.
5. **Generación de respuestas con un LLM (Qwen 4-bit)**.
6. **Validación y métricas**: grounding score, BLEU, ROUGE, y citas por oración.

El objetivo es **responder preguntas usando solo la información contenida en los documentos**, minimizando alucinaciones y asegurando trazabilidad de las respuestas.

### 0. Instalación de librerías
Se instalan todas las dependencias necesarias:
- `transformers`, `accelerate`, `bitsandbytes`, `sentencepiece` → para LLM y cuantización 4-bit.
- `faiss-cpu`, `sentence-transformers`, `datasets` → para embeddings y FAISS.
- `rank_bm25`, `rouge_score`, `nltk`, `pypdf` → para recuperación híbrida, evaluación automática y manejo de PDFs.

Garantiza que todo el pipeline (RAG, reranking, métricas, LLM) pueda ejecutarse sin errores por falta de librerías.

### Instalación de librerías

In [ ]:
!pip install -U transformers accelerate bitsandbytes sentencepiece
!pip install faiss-cpu sentence-transformers datasets
!pip install rank_bm25 rouge_score nltk pypdf

...instalación completada...


### 1. Preparación de carpeta de documentos
Se crea la carpeta `/content/docs` donde se subirán los PDFs que formarán el dataset. Esto asegura que el pipeline RAG tenga un lugar definido para leer los documentos.

In [ ]:
# comandos para crear un carpeta en los contenidos de nombre "docs", donde almacenaremos los documentos
import os
os.makedirs("/content/docs", exist_ok=True)
print("Carpeta lista — sube tus PDFs a la carpeta docs en el panel de archivos")

Carpeta lista — sube tus PDFs a la carpeta docs en el panel de archivos


### 2. Dataset (Carga de PDFs)
Se leen los PDFs de la carpeta, página por página, limpiando saltos de línea y espacios. Cada página se guarda en `texts` y se le asigna un título en `titles` Esto prepara los documentos para el pipeline RAG. Se imprime la cantidad de páginas cargadas y un ejemplo de contenido.

In [ ]:
from pypdf import PdfReader
import re, os

texts, titles = [], []

for filename in os.listdir("/content/docs"):
    if not filename.endswith(".pdf"):
        continue
    reader   = PdfReader(f"/content/docs/{filename}")
    doc_name = filename.replace(".pdf", "")

    for i, page in enumerate(reader.pages):
        texto = page.extract_text() or ""
        texto = re.sub(r"\s+", " ", texto.replace("\n", " ")).strip()
        if len(texto) > 100:
            texts.append(texto)
            titles.append(f"{doc_name} · p{i+1}")

print("Páginas cargadas:", len(texts))
print("Ejemplo —", titles[0], ":", texts[0][:200])

Páginas cargadas: 850
Ejemplo — oracle-introduccion · p1 : Manual Curso Introductorio...


### 3. Chunking con Overlap

En esta sección se divide el texto en fragmentos (chunks) con solapamiento(overlap) para mantener continuidad entre ellos.

**Tradeoff (equilibrio) del tamaño de chunk**
- Chunk pequeño → búsqueda precisa, pero puede perder contexto
- Chunk grande → más contexto para el LLM, pero menos precisión al buscar

**El Tradeoff (equilibrio) del overlap:**
- Chunk pequeño → búsqueda precisa, pero puede perder contexto
- Chunk grande → más contexto para el LLM, pero menos precisión al buscar

Se utiliza `size=400, overlap=100`  como un equilibrio entre contexto, precisión y eficiencia.

In [ ]:
def chunk_text(text, title, size=400, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append({"text": text[start:start+size], "title": title})
        start += size - overlap
    return chunks

all_chunks = []
for text, title in zip(texts, titles):
    all_chunks.extend(chunk_text(text, title))

chunk_texts  = [c["text"]  for c in all_chunks]
chunk_titles = [c["title"] for c in all_chunks]

print("Total chunks:", len(all_chunks))

Total chunks: 4964


### 4. Embeddings

Se convierten los chunks de texto en vectores numéricos (embeddings) usando un modelo pre-entrenado. Cada vector representa el significado de un chunk, lo que permite comparar y buscar información semántica de manera eficiente. (Textos con significado similar quedan cerca en el espacio vectorial).

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder   = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = embedder.encode(chunk_texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

print("Shape:", embeddings.shape)

Shape: (4964, 384)


### 5. FAISS HNSW
Se utiliza **FAISS** con **HNSW** (Hierarchical Navigable Small World) para indexar los embeddings de los chunks y permitir búsquedas semánticas rápidas.

**HNSW** organiza los vectores en capas jerárquicas:
- Navega rápido por capas altas y refina en capas bajas, como un GPS.
- Ventaja sobre IVFFlat: no requiere entrenamiento previo y ofrece mejor recall.

**Parámetros importantes**:
- `M=32` → conexiones por nodo (más = mejor calidad, más memoria)
- `efConstruction=200` → esfuerzo al construir
- `efSearch=50` → esfuerzo al buscar vecinos

In [ ]:
import faiss

dim = embeddings.shape[1]
faiss.normalize_L2(embeddings)

hnsw = faiss.IndexHNSWFlat(dim, 32)
hnsw.hnsw.efConstruction = 200
hnsw.hnsw.efSearch = 50
hnsw.add(embeddings)

print("Vectores indexados:", hnsw.ntotal)

Vectores indexados: 4964


### 6. Búsqueda Híbrida: BM25 + FAISS

Para mejorar la recuperación de información se combina:

-**BM25** = búsqueda clásica por palabras clave.

-**FAISS** = búsqueda semántica basada en embeddings (por significado).

La combinación se hace mediante una mezcla lineal controlada por un parámetro alpha:

**score = alpha × FAISS + (1 - alpha) × BM25**

**FAISS** y **BM25** se normalizan entre 0 y 1 para que los scores sean comparables.

- alpha decide la importancia relativa de cada método:

    (i) alpha cercano a 1 → más peso a FAISS (semántica)

    (ii) alpha cercano a 0 → más peso a BM25 (palabras clave)

- Ejemplo típico: score = 0.7 × FAISS + 0.3 × BM25

**Ventajas:**

- Cubre las limitaciones de cada método: FAISS maneja sinónimos y contexto BM25 garantiza coincidencias literales precisas.

- **La función retrieve()** devuelve los k chunks más relevantes según la consulta, integrando semántica y palabras clave de manera eficiente.

In [ ]:
from rank_bm25 import BM25Okapi

tokenized = [t.lower().split() for t in chunk_texts]
bm25      = BM25Okapi(tokenized)

def retrieve(query, k=10, alpha=0.7):
    n = len(chunk_texts)
    q_emb = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    D, I  = hnsw.search(q_emb, min(k*3, n))
    faiss_scores = np.zeros(n)
    for s, i in zip(D[0], I[0]):
        if i >= 0: faiss_scores[i] = float(s)
    if faiss_scores.max() > 0:
        faiss_scores /= faiss_scores.max()
    bm25_raw = bm25.get_scores(query.lower().split())
    bm25_scores = bm25_raw / bm25_raw.max() if bm25_raw.max() > 0 else bm25_raw
    hybrid  = alpha * faiss_scores + (1 - alpha) * bm25_scores
    top_idx = np.argsort(hybrid)[::-1][:k]
    return [{"text": chunk_texts[i], "title": chunk_titles[i]} for i in top_idx]

### 7. Reranker Cross-Encoder

FAISS usa un **bi-encoder**: codifica pregunta y documento por separado, luego compara vectores. Rápido pero aproximado.

El reranker usa un **cross-encoder**: analiza pregunta + documento juntos. Mucho más preciso.

Estrategia del pipeline:
1. FAISS recupera top-10 chunks rápido (búsqueda semántica aproximada).
2. El reranker reordena esos top-10 usando cross-encoder para evaluar relevancia exacta.
3. Se seleccionan los top-3 resultados para alimentar al LLM.

**Función rerank():**
- Recibe una consulta y una lista de chunks candidatos.
- Predice un score de relevancia para cada par (consulta, chunk).
- Devuelve los top-k candidatos ordenados por relevancia.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=3):
    scores = reranker.predict([(query, c["text"]) for c in candidates])
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

Reranker cargado


### 8. LLM Qwen (4-bit)
El LLM es el modelo final que genera la respuesta basada en los chunks más relevantes.

**Características:**
- Qwen 1.5 (~1.8B parámetros), cargado con cuantización de 4 bits. Esto reduce su tamaño de ~7GB a ~1GB de VRAM, permitiendo usar GPUs como la T4 de Colab.
- El system prompt le indica que responda **solo con el contexto** proporcionado. Esto es la primera defensa contra alucinaciones.

**Componentes:**
- torch → backend PyTorch para operaciones tensoriales y GPU.
- AutoTokenizer → convierte texto a tokens que el modelo entiende.
- AutoModelForCausalLM → modelo de lenguaje causal capaz de generar texto.
- BitsAndBytesConfig → configuración para carga en 4 bits y ahorro de memoria.

**Flujo en el pipeline:**
1. Recibe los top-k chunks seleccionados por FAISS + Reranker.
2. Genera la respuesta textual final respetando el contexto proporcionado.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen1.5-1.8B-Chat"

bnb       = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("✅ Qwen cargado")

✅ Qwen cargado


### 9. Prompt + generación de respuestas
Este paso utiliza el LLM (Qwen 4-bit) para generar la respuesta final basada en los chunks más relevantes obtenidos del pipeline RAG.

Flujo:
1. Recibe la pregunta del usuario y el contexto (chunks top después de retrieval + reranker).
2. Construye un prompt estructurado.
3. Convierte el prompt a tokens usando el tokenizer.
4. Genera texto con el modelo Qwen (quantized 4-bit para eficiencia).
5. Devuelve la respuesta textual final.

Importancia:
- Garantiza que el LLM solo use información recuperada (previene alucinaciones).
- Completa la etapa de generación del pipeline RAG.

In [ ]:
def ask_llm(context, question):
    messages = [
        {"role": "system", "content": "Responde ÚNICAMENTE usando el contexto. Si no está, di 'No encontrado en el contexto'."},
        {"role": "user",   "content": f"Contexto:\n{context}\n\nPregunta:\n{question}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

### 10. Citation per sentence

Este paso mejora la trazabilidad de la respuesta del LLM, agregando referencias a los chunks originales que respaldan cada oración de la respuesta.

**Flujo:**
1. Divide la respuesta generada en oraciones.
2. Para cada oración, calcula el "overlap" de palabras con cada chunk.
3. Asigna como cita el título del chunk con mayor overlap.
4. Si no hay coincidencia significativa (>10% palabras), marca "no encontrado".
5. Devuelve la respuesta con citas por oración.

In [ ]:
import re

def cite_sentences(answer, chunks):
    sentences = re.split(r'(?<=[.!?])\s+', answer.strip())
    sentences = [s for s in sentences if len(s) > 10]
    result = []
    for sent in sentences:
        words = set(sent.lower().split())
        best, best_score = "no encontrado", -1
        for c in chunks:
            overlap = len(words & set(c["text"].lower().split())) / max(len(words), 1)
            if overlap > best_score:
                best_score, best = overlap, c["title"]
        citation = best if best_score > 0.1 else "no encontrado"
        result.append(f"{sent} [{citation}]")
    return "\n".join(result)

### 11. Hallucination guard + grounding score

Este paso verifica que la respuesta del LLM esté respaldada por el contexto.

Flujo:
1. Extrae todas las palabras de los chunks recuperados.
2. Filtra palabras irrelevantes (stopwords y cortas) de la respuesta.
3. Calcula el grounding score: proporción de palabras de la respuesta presentes en el contexto.
4. Asigna un veredicto según el score:
- ≥0.7 ✅ FUNDAMENTADO
- 0.4–0.7 ⚠️ ADVERTENCIA
- <0.4 ❌ ALUCINACIÓN

In [ ]:
def hallucination_guard(answer, chunks):
    ctx_words = set(" ".join(c["text"] for c in chunks).lower().split())
    stopwords = {"el","la","los","las","un","una","es","son","en","de","y","a","the","is","of","and","to","a"}
    content   = [w for w in answer.lower().split() if w not in stopwords and len(w) > 2]
    if not content:
        return 0.0, "SIN DATOS"
    score   = sum(1 for w in content if w in ctx_words) / len(content)
    verdict = "✅ FUNDAMENTADO" if score >= 0.7 else ("⚠️ ADVERTENCIA" if score >= 0.4 else "❌ ALUCINACIÓN")
    return round(score, 4), verdict

### 12. BLEU + ROUGE
Este paso permite medir cuantitativamente la calidad de la respuesta generada comparándola con una referencia.

Flujo:
1. Tokeniza y normaliza a minúsculas la respuesta generada y la referencia.
2. Calcula BLEU-1 → mide coincidencia de palabras individuales (unigrams).
3. Calcula ROUGE-L → mide coincidencia de secuencias largas (longest common subsequence).
4. Devuelve un diccionario con BLEU-1 y ROUGE-L redondeados a 4 decimales.

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

rouge    = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
smoother = SmoothingFunction().method1

def evaluate(generated, reference):
    ref   = reference.lower().split()
    gen   = generated.lower().split()
    bleu1 = sentence_bleu([ref], gen, weights=(1,0,0,0), smoothing_function=smoother)
    rs    = rouge.score(reference, generated)
    return {
        "BLEU-1":  round(bleu1, 4),
        "ROUGE-L": round(rs["rougeL"].fmeasure, 4),
    }

### 13. Consulta
Aquí el usuario hace la pregunta y se ejecuta todo el pipeline: retrieve → rerank → LLM → hallucination guard.

(i) Se obtiene el contexto de los chunks más relevantes.

(ii) El LLM genera la respuesta usando solo ese contexto.

(iii) Se calcula el grounding score para medir si la respuesta está fundamentada.

**Objetivo:** devolver una respuesta confiable basada en los documentos originales.

In [ ]:
query = "¿Qué es un tablespace en Oracle?"  # ← cambia por tu pregunta

docs    = retrieve(query, k=10)
top     = rerank(query, docs, top_k=3)
context = "\n".join(c["text"] for c in top)
answer  = ask_llm(context, query)
score, verdict = hallucination_guard(answer, top)

print("RESPUESTA:\n", answer)
print("\nGrounding score:", score, verdict)

RESPUESTA:
 Un tablespace en Oracle es una unidad lógica básica de memoria...

Grounding score: 0.2917 ❌ ALUCINACIÓN


**concluimos lo siguiente:**

El grounding score de 0.2917 indica que menos del 30% de las palabras clave de la respuesta coinciden con los chunks seleccionados del contexto.

El veredicto ❌ ALUCINACIÓN significa que la respuesta contiene información no fundamentada en los documentos que proporcionaste.

Aunque la respuesta suena correcta sobre Oracle, tu pipeline detectó que gran parte del contenido no estaba respaldado por los documentos indexados, lo que puede generar errores o imprecisiones.

### Varias consultas

In [ ]:
questions = [
    ("¿Cuáles son las tareas básicas de un DBA?",     "El DBA debe instalar software, monitorear el sistema y garantizar la seguridad."),
    ("¿Qué es un índice en Oracle?",                  "Un índice es una estructura que permite recuperar datos de forma más rápida."),
    ("What is linear algebra?", "Linear algebra studies vectors, matrices and linear transformations."),
]

results = []

for query, reference in questions:
    docs    = retrieve(query, k=10)
    top     = rerank(query, docs, top_k=3)
    context = "\n".join(c["text"] for c in top)
    answer  = ask_llm(context, query)
    cited   = cite_sentences(answer, top)
    score, verdict = hallucination_guard(answer, top)
    metrics = {"grounding": score, "verdict": verdict}
    metrics.update(evaluate(answer, reference))
    results.append({"query": query, "answer": answer, "cited": cited, "metrics": metrics})

    print("="*60)
    print("Q:", query)
    print("\nRespuesta con citas:")
    print(cited)
    print("\nMétricas:", metrics)

...resultados de varias consultas...


### Reporte final

In [ ]:
print(f"{'='*65}")
print(f"{'Pregunta':<30} {'Grounding':>10} {'BLEU-1':>8} {'ROUGE-L':>8}  Veredicto")
print("-"*65)
for r in results:
    q  = r['query'][:28] + ".." if len(r['query']) > 28 else r['query']
    g  = r['metrics']['grounding']
    b1 = r['metrics'].get('BLEU-1', '-')
    rl = r['metrics'].get('ROUGE-L', '-')
    v  = r['metrics']['verdict']
    print(f"{q:<30} {g:>10.4f} {str(b1):>8} {str(rl):>8}  {v}")
print("="*65)

Pregunta                        Grounding   BLEU-1  ROUGE-L  Veredicto
-----------------------------------------------------------------
¿Cuáles son las tareas básic..     0.4023   0.0552    0.071  ⚠️ ADVERTENCIA
¿Qué es un índice en Oracle?       0.4476   0.0533    0.104  ⚠️ ADVERTENCIA
What is linear algebra?            0.0192   0.0144   0.0248  ❌ ALUCINACIÓN
